# RetinAI — EfficientNet-B4 Training on Kaggle GPU
### SIH 2026 Problem Statement: SIH26038 (MathWorks)
**Explainable AI for Diabetic Retinopathy Screening**

This notebook trains an **EfficientNet-B4** model with:
- Quadratic Weighted Kappa (QWK) optimization
- Class-weighted CrossEntropyLoss for severe class imbalance
- Cosine Annealing Learning Rate Schedule
- Automatic Mixed Precision (AMP / FP16) for fast training (~30-40 mins on GPU)

## 1. Install Dependencies

In [ ]:
!pip install timm albumentations scikit-learn -q

## 2. Training Pipeline

In [ ]:
import os
import gc
import time
import glob
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, classification_report
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Config ────────────────────────────────────────────────────
CFG = {
    "model_name":   "efficientnet_b4",
    "img_size":     380,
    "num_classes":  5,
    "epochs":       12,
    "batch_size":   16,
    "lr":           3e-4,
    "min_lr":       1e-6,
    "weight_decay": 1e-4,
    "seed":         42,
    "device":       "cuda" if torch.cuda.is_available() else "cpu",
    "output_dir":   "/kaggle/working",
}

print(f"Device: {CFG['device']} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# ── Seed ─────────────────────────────────────────────────────
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])

# ── Auto-Detect Dataset Location ─────────────────────────────
possible_dirs = [
    "/kaggle/input/aptos2019-blindness-detection",
    "/kaggle/input/competitions/aptos2019-blindness-detection",
    os.path.expanduser("~/.cache/kagglehub/competitions/aptos2019-blindness-detection"),
]

DATA_DIR = None
for p in possible_dirs:
    if os.path.exists(p):
        for root, dirs, files in os.walk(p):
            if "train.csv" in files:
                DATA_DIR = root
                break
    if DATA_DIR:
        break

if not DATA_DIR:
    DATA_DIR = "/kaggle/input/aptos2019-blindness-detection"

print(f"Dataset root: {DATA_DIR}")

# ── Dataset Definition ────────────────────────────────────────
class APTOSDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["id_code"]
        path = os.path.join(self.img_dir, f"{img_id}.png")
        if not os.path.exists(path):
            path = os.path.join(self.img_dir, f"{img_id}.jpg")
        img = np.array(Image.open(path).convert("RGB"))
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, int(row["diagnosis"])

# ── Augmentations ─────────────────────────────────────────────
def get_train_transform(img_size):
    return A.Compose([
        A.RandomResizedCrop(img_size, img_size, scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=30, p=0.5),
        A.CLAHE(clip_limit=2.0, p=0.4),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, p=0.4),
        A.GaussNoise(p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_val_transform(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

# ── Training Loops ────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss, total_samples = 0.0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast(enabled=(device == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        if device == "cuda":
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * len(labels)
        total_samples += len(labels)
    return total_loss / total_samples

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with autocast(enabled=(device == "cuda")):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            total_loss += loss.item() * len(labels)
            total_samples += len(labels)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    kappa = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
    return total_loss / total_samples, kappa, all_preds, all_labels

# ── Load Data & Start Training ───────────────────────────────
train_csv_path = os.path.join(DATA_DIR, "train.csv")
if not os.path.exists(train_csv_path):
    train_csv_path = glob.glob(f"{DATA_DIR}/**/train.csv", recursive=True)[0]

train_df = pd.read_csv(train_csv_path)
img_dir = os.path.join(os.path.dirname(train_csv_path), "train_images")
if not os.path.exists(img_dir):
    img_dir = os.path.dirname(train_csv_path)

grade_names = ["No DR", "Mild DR", "Moderate DR", "Severe DR", "Proliferative DR"]
print(f"Total images: {len(train_df)}")

counts = train_df["diagnosis"].value_counts().sort_index().values.astype(float)
weights = torch.tensor(1.0 / counts, dtype=torch.float)
weights = (weights / weights.sum()) * len(counts)
weights = weights.to(CFG["device"])
criterion = nn.CrossEntropyLoss(weight=weights)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG["seed"])
train_idx, val_idx = next(iter(skf.split(train_df, train_df["diagnosis"])))

train_ds = APTOSDataset(train_df.iloc[train_idx], img_dir, get_train_transform(CFG["img_size"]))
val_ds   = APTOSDataset(train_df.iloc[val_idx],   img_dir, get_val_transform(CFG["img_size"]))

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

model = timm.create_model(CFG["model_name"], pretrained=True, num_classes=CFG["num_classes"]).to(CFG["device"])
optimizer = optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"], eta_min=CFG["min_lr"])
scaler = GradScaler(enabled=(CFG["device"] == "cuda"))

best_kappa = -1.0
best_model_path = os.path.join(CFG["output_dir"], "best_dr_model.pth")

print("\nStarting training...")
for epoch in range(CFG["epochs"]):
    ep_start = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, CFG["device"])
    val_loss, kappa, _, _ = val_epoch(model, val_loader, criterion, CFG["device"])
    scheduler.step()
    
    saved_tag = ""
    if kappa > best_kappa:
        best_kappa = kappa
        torch.save(model.state_dict(), best_model_path)
        saved_tag = "  --> [SAVED NEW BEST]"
        
    print(f"Epoch [{epoch+1:02d}/{CFG['epochs']:02d}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | QWK: {kappa:.4f} | Time: {time.time()-ep_start:.1f}s{saved_tag}")

print(f"\nTraining complete! Best Quadratic Weighted Kappa: {best_kappa:.4f}")
print(f"Model saved at: {best_model_path}")

## 3. Verify Saved Model Output

In [ ]:
import os
file_size_mb = os.path.getsize("/kaggle/working/best_dr_model.pth") / (1024 * 1024)
print(f"✓ best_dr_model.pth is ready for download! Size: {file_size_mb:.2f} MB")